# Microsoft Agent Framework: Human in the Loop

Sometimes an automated workflow should **pause and ask you** before continuing. This is human in the loop (HITL): a person reviews, corrects, approves, or rejects a proposed action.

This beginner lab uses no model or API key. You will learn `@workflow`, `@step`, `request_info()`, pending request events, and resume with `responses`.

## Real-life analogy

Imagine a publishing team that checks with an editor:

- the workflow prepares a draft,
- it pauses and places the draft in the editor's inbox,
- the editor supplies feedback, and
- the workflow resumes without repeating the earlier work.

## 1. Setup

Use Python 3.10+. If you installed the repository's `requirements.txt`, skip the next cell. Otherwise, uncomment it and run it once.

In [ ]:
# %pip install -q agent-framework-core==1.19.0

## 2. Build a review workflow

`request_info()` emits a request and pauses. The host—not the workflow—collects the human response. Completed `@step` calls are saved, so the draft step does not run again after resume.

In [ ]:
from agent_framework import RunContext, WorkflowRunState, step, workflow

execution_counts = {"draft": 0, "finalize": 0}

@step
async def write_draft(topic: str) -> str:
    execution_counts["draft"] += 1
    print(f"Writing draft (execution #{execution_counts['draft']})")
    return f"Draft: A short introduction to {topic}."

@step
async def apply_review(draft: str, feedback: str) -> str:
    execution_counts["finalize"] += 1
    return f"{draft}\nHuman feedback: {feedback}"

@workflow
async def review_workflow(topic: str, ctx: RunContext) -> str:
    draft = await write_draft(topic)
    feedback = await ctx.request_info(
        {"draft": draft, "question": "What should change?"},
        response_type=str,
        request_id="editor_review",
    )
    return await apply_review(draft, feedback)

## 3. Run until the pause

The first run stops in `IDLE_WITH_PENDING_REQUESTS`. Inspect the event before deciding what your UI should show the reviewer.

In [ ]:
instance = review_workflow.build()
first = await instance.run("responsible AI")

print("State:", first.get_final_state())
assert first.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS

pending = first.get_request_info_events()
for event in pending:
    print("Request id:", event.request_id)
    print("Payload:", event.data)

## 4. Simulate the human and resume

A real app would get this response from an authenticated reviewer. A fixed response keeps the lesson reproducible.

In [ ]:
reviewer_feedback = "Add one concrete example and define the acronym."
final = await instance.run(responses={pending[0].request_id: reviewer_feedback})

print("State:", final.get_final_state())
print("Output:\n", final.get_outputs()[0])
print("Step executions:", execution_counts)
assert execution_counts == {"draft": 1, "finalize": 1}

## 5. Try a real interactive review

Build a fresh instance because the previous one completed.

In [ ]:
interactive_instance = review_workflow.build()
paused = await interactive_instance.run("agent safety")
request = paused.get_request_info_events()[0]
print(request.data["draft"])
human_feedback = input("Your feedback: ") or "Looks good"
completed = await interactive_instance.run(
    responses={request.request_id: human_feedback}
)
print(completed.get_outputs()[0])

## Tool approval versus free-form review

Use `request_info()` when a human must add information or judgment. For a concrete model-proposed side effect, mark the tool with `@tool(approval_mode="always_require")`. The agent result exposes `user_input_requests`; the host inspects the function call and creates a response with `to_function_approval_response(True/False)`. See `examples/agent_tool_approval.py`.

## Key points to remember

- HITL is a control boundary, not merely a text box.
- Show the exact action and arguments to the reviewer.
- Keep the request id and send the response back with that id.
- Use `@step` so resume does not repeat expensive work.
- Authenticate reviewers, expire approvals, log decisions, and make side effects idempotent.

## Exercises

1. Change the response to a Pydantic model with `approved` and `comments`.
2. Add a rejection branch.
3. Replace `write_draft` with an agent call.
4. Store reviewer identity, timestamp, request id, and decision in an audit table.
5. Add durable checkpoint storage and resume after a process restart.

## References

- [MAF human-in-the-loop workflows](https://learn.microsoft.com/en-us/agent-framework/workflows/human-in-the-loop)
- [MAF checkpoints](https://learn.microsoft.com/en-us/agent-framework/workflows/checkpoints)
- [MAF tool approvals](https://learn.microsoft.com/en-us/agent-framework/agents/tools/tool-approval)
- [Official MAF Python samples](https://github.com/microsoft/agent-framework/tree/main/python/samples)